Tutorial at https://docs.google.com/document/d/1CcEuQ653DHJ0iO9pg0PC34Acx84sp44umAeQXu6cq5A/edit?usp=sharing

In [ ]:
# Path to the lif that contains the chip to build the new cropping profile from
lf = r"C:\Users\rappez\Desktop\data_temp\20250512_tulip12thaw_pffb_10_d4_Lucapar.lif"
profile_name = 'dorota_test' # name of teh enw cropping profile - will be used later in teh vascumap pipeline
img_index = 1 # index of the image in the lif file
crop_profile_folder = r"C:\ANALYSIS\crop_profiles" # Folder containing the cropping profiles

import os
if os.path.exists(os.path.join(crop_profile_folder, f'{profile_name}_fiducials.npy')):
    print(f'{profile_name}_fiducials.npy already exists, chose another profile name')

In [ ]:
os.chdir(r"C:\Users\rappez\Documents\python_codebase")
from VascuMap2D.src.utils import auto_focus
from VascuMap3D.scripts.whole_lif_inference_local import get_stack_from_lif
from aicsimageio import AICSImage
import napari
import numpy as np

def interpolate_coordinates(coord_arrays, spacing=1.0):
    """
    Linearly interpolate between consecutive coordinates with unit spacing.
    
    Parameters:
    - coord_arrays: List of NumPy arrays with shape (N, 2)
    - spacing: Distance between interpolated points (default=1.0)
    
    Returns:
    - interpolated_list: List of arrays with interpolated coordinates
    """
    interpolated_list = []

    for coords in coord_arrays:
        interpolated = []
        for i in range(len(coords) - 1):
            start = coords[i]
            end = coords[i + 1]
            
            # Calculate segment length
            segment_vector = end - start
            segment_length = np.linalg.norm(segment_vector)
            
            # Calculate number of points needed for this segment
            num_points = max(2, int(np.ceil(segment_length / spacing)))
            
            # Create points along the segment
            for t in np.linspace(0, 1, num_points, endpoint=False):
                point = start + t * segment_vector
                interpolated.append(point)
        
        # Add the last point explicitly to close the segment
        interpolated.append(coords[-1])
        interpolated_list.append(np.array(interpolated, dtype=np.float32))

    return interpolated_list

img = AICSImage(lf)
channel_names = img.channel_names
bf_index = [i for i, name in enumerate(channel_names) if 'BF' in name or 'Gray' in name][0]
stack, voxel_size, shape = get_stack_from_lif(lf, stack_index=img_index, ref_voxel_size=[5, 2, 2])

nz, ny, nx, nc = stack.shape

crop_window = 0.20*nx
if crop_window < 200:
    crop_window = 200

focus, grid = auto_focus(
    stack_bf=stack[..., bf_index], 
    viz=False,
    crop_window=crop_window, 
    n_sampling=5, 
    window_size=400
    )

v = napari.Viewer()
v.add_image(focus)

In [ ]:
import matplotlib.pyplot as plt

neck_coords = v.layers['Shapes'].data
rect_coords = np.array(v.layers['Shapes [1]'].data)[0]
neck_coords_interpolated = [interpolate_coordinates([arr], spacing=1.0)[0] for arr in neck_coords]
neck_coords_all = np.vstack(neck_coords_interpolated)  # shape (M, 2)

plt.figure()
plt.imshow(focus, cmap='gray')
plt.scatter(neck_coords_all[:, 1], neck_coords_all[:, 0], c='r')
plt.plot(rect_coords[:, 1], rect_coords[:, 0], color=[0,1,0])
plt.plot(rect_coords[[0,-1], 1], rect_coords[[0,-1], 0], color=[0,1,0])
plt.axis('equal')
plt.show()

In [ ]:
if os.path.exists(os.path.join(crop_profile_folder, f'{profile_name}_fiducials.npy')):
    print(f'{profile_name}_fiducials.npy already exists, chose another profile name')
else:
    np.save(os.path.join(crop_profile_folder, f'{profile_name}_fiducials.npy'), neck_coords_all)
    np.save(os.path.join(crop_profile_folder, f'{profile_name}_rect.npy'), rect_coords)